# EXAONE 모델을 사용한 질문-답변 시스템

현재 Colab에 설치된 transformers 패키지의 버전은 5.0.0

EXAONE 모델을 사용하기 위해서는 5.1.0 버전이 필요.

In [1]:
!pip install -U transformers==5.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 55.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


Imports

In [2]:
import numpy as np
import scipy
import transformers

In [3]:
transformers.__version__

'5.1.0'

In [4]:
# 깃허브에 ipynb 파일을 업로드할 때 다운로드 상태(진행바) 표시줄때문에 오류 발생
# -> 깃허브에 정상적으로 업로드되게 하기 위해서
transformers.utils.logging.disable_progress_bar()

# EXAONE-3.5 모델

In [5]:
pipe = transformers.pipeline(
    task='text-generation',
    model='LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct',
    device=0,  # GPU 사용
    trust_remote_code=True
)

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [21]:
# 메시지 템플릿: 'role'과 'content'를 키로 갖는 dict들을 아이템으로 갖는 리스트.
# role: 역할(system, user, assistant)
# content: 내용
messages = [
    {
        'role': 'system',
        'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.\
        확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는 \
        간단하고 친절한 답변을 생성해줘.'
    },
    {
        'role': 'user',
        'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'
    }
]

In [22]:
result = pipe(messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [23]:
print(result)
#> pipeline 호출 결과: 'generated_text' 키를 갖는 dict 1개를 저장하고 있는 리스트

[{'generated_text': [{'role': 'system', 'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.        확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'}, {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'}, {'role': 'assistant', 'content': '네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 미리 준비되어 있어서 편리하게 계획을 세울 수 있도록 도와드리고 있습니다. 하지만 자세한 사항이나 날짜 확인은 직접 제품 담당자에게 문의하시는 게 가장 정확할 것 같아요. 담당자분께서 즉시 답변을 드릴 수 있도록 연락주시면 감사하겠습니다! 😊'}]}]


In [24]:
len(result)

1

In [25]:
result[0]

{'generated_text': [{'role': 'system',
   'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.        확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'},
  {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'},
  {'role': 'assistant',
   'content': '네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 미리 준비되어 있어서 편리하게 계획을 세울 수 있도록 도와드리고 있습니다. 하지만 자세한 사항이나 날짜 확인은 직접 제품 담당자에게 문의하시는 게 가장 정확할 것 같아요. 담당자분께서 즉시 답변을 드릴 수 있도록 연락주시면 감사하겠습니다! 😊'}]}

In [26]:
result[0].keys()

dict_keys(['generated_text'])

result - `[ { `generated_text`:[...] } ]`

In [27]:
result[0]['generated_text']
#> 'role'과 'content'를 키로 갖는 dict들의 list.
#> 3개의 dict: 첫 2개는 입력값, 마지막 1개 AI가 생성한 답변

[{'role': 'system',
  'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.        확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'},
 {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'},
 {'role': 'assistant',
  'content': '네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 미리 준비되어 있어서 편리하게 계획을 세울 수 있도록 도와드리고 있습니다. 하지만 자세한 사항이나 날짜 확인은 직접 제품 담당자에게 문의하시는 게 가장 정확할 것 같아요. 담당자분께서 즉시 답변을 드릴 수 있도록 연락주시면 감사하겠습니다! 😊'}]

In [28]:
result[0]['generated_text'][-1]

{'role': 'assistant',
 'content': '네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 미리 준비되어 있어서 편리하게 계획을 세울 수 있도록 도와드리고 있습니다. 하지만 자세한 사항이나 날짜 확인은 직접 제품 담당자에게 문의하시는 게 가장 정확할 것 같아요. 담당자분께서 즉시 답변을 드릴 수 있도록 연락주시면 감사하겠습니다! 😊'}

In [29]:
result[0]['generated_text'][-1]['content']

'네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 미리 준비되어 있어서 편리하게 계획을 세울 수 있도록 도와드리고 있습니다. 하지만 자세한 사항이나 날짜 확인은 직접 제품 담당자에게 문의하시는 게 가장 정확할 것 같아요. 담당자분께서 즉시 답변을 드릴 수 있도록 연락주시면 감사하겠습니다! 😊'

pipeline은 실행할 때마다 다른 답변(텍스트)를 생성함.

pipeline의 파라미터:

*   `return_full_text`: 이전의 대화 기록을 모두 리턴할 여부. 기본값은 True.
*   `do_sample`: 샘플링 전략을 사용할 지 여부를 설정. 기본값은 True.
    *   `do_sample=True`: 메시지 프롬프트가 같아도 실행할 때마다 생성되는 텍스트가 달라짐.
    *   `do_sample=False`
        *   메시지 프롬프트가 같으면 항상 같은 텍스트가 생성됨.
        *   가장 확률이 높은 토큰들만 선택해서 텍스트를 생성.
*   샘플링 전략
    *  temperature(온도)
    *   top-k 방식
    *   top-p 방식

In [30]:
result = pipe(messages,
              max_new_tokens=200,    return_full_text=False, do_sample=False)
print(result)
#> result - [ { 'generated_text': '답변'} ]
#> 실행할 때마다 항상 같은 답변.

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 궁금하시군요. 제품 담당자분께서 가장 정확한 정보를 제공해 드릴 수 있을 것 같아요. 그분께서는 다이어리의 최신 업데이트 내용과 내년 공휴일 정보를 확인하실 수 있으니, 조금만 기다려 주시면 곧 답변을 드릴 수 있을 것 같습니다. 궁금한 점이 더 있으시면 언제든지 알려주세요! 😊'}]


# 샘플링 전략 - temperature

In [32]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=10.0)
print(result)

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '당신계해 오신 궁금짐에도 제 지식베이스 만선에는 정확히 기재한다이어리 페이지 구성 옵션, 즉 특별해 한가지가 포함하여 다이어리내년휴일 표시와 있기 라는 사항도 포함되도록 세세정시한지 명확히 하지 없곤 못합니다... 저희 사무실의 팀께 질문 남겨확인 하셔봐도 더 신속정확합답들킬 텐데 혹시 가능한 건가요? 함께 상의하면서 빠르게더 해결책 구하기를 바랄게요. 빠르실 경우 제 추천 사항들이 크롭히게 쓰일수음으므로 그 시간 낭비 방지용이지시 알려달라 부탁드려요!\\ _\\\\ **빠른응답과 맞춤 지원 필요시 관리자로 신고 도와드릴테니아드리길 권장** 💭✨ⱹ 확인 위해 바로 바로 피드백을 해 드리도록 할게요.~ 🤬 😡�\u200b\u200b\u200b  도움주시거나 알려주간을 해 줄게니 지금도 진행시켜주겠냿가요~ 💝\\*\\* 8시 마감까제주세** ⚂**** 💧 💋 💟\u200d� Dalton'}]


In [33]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=0.001)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 궁금하시군요. 제품 담당자분께서 가장 정확한 정보를 제공해 드릴 수 있을 것 같아요. 그분께서는 다이어리의 최신 업데이트 내용과 내년 공휴일 정보를 확인하실 수 있으니, 조금만 기다려 주시면 곧 답변을 드릴 수 있을 것 같습니다. 궁금한 점이 더 있으시면 언제든지 알려주세요! 😊'}]


temperature(온도)를 1보다 큰 값을 사용하면 문장을 생성할 때 선택되는 토큰들이 다양해 지기 때문에 답변이 무작위짐.

temperature를 1보다 작은 값을 사용하면 높은 확률을 가진 토큰들이 선택될 가능성이 더 커짐. 생성되는 답변이 더 일관되어 짐.

In [37]:
# 동전 던지기
probs = [0.5, 0.5]  # 앞면 확률, 뒷면 확률

np.random.multinomial(n=100, pvals=probs)  # 동전 던지기 100번 실험

array([53, 47])

In [39]:
# 주사위 던지기
probs = [1/6] * 6
np.random.multinomial(n=600, pvals=probs)

array([102,  89,  96,  99, 104, 110])

logit(로짓): softmax 함수의 아규먼트. LLM에서는 어휘 사전에 포함된 각 토큰에 대한 점수.

In [40]:
logits = [1, 2, 3, 4, 10]  # 5개 토큰의 점수

In [41]:
probs = scipy.special.softmax(logits)
print(probs)

[1.22936559e-04 3.34176214e-04 9.08385131e-04 2.46924679e-03
 9.96165255e-01]


In [46]:
np.random.multinomial(n=200, pvals=probs)

array([  0,   0,   0,   2, 198])

In [49]:
logits = np.array([1, 2, 3, 4, 100])
probs = scipy.special.softmax(logits)
print(probs)

[1.01122149e-43 2.74878501e-43 7.47197234e-43 2.03109266e-42
 1.00000000e+00]


In [50]:
np.random.multinomial(n=200, pvals=probs)

array([  0,   0,   0,   0, 200])

In [51]:
probs = scipy.special.softmax(logits / 10)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[5.01629119e-05 5.54385914e-05 6.12691190e-05 6.77128484e-05
 9.99765417e-01]


array([  0,   0,   0,   0, 200])

In [52]:
probs = scipy.special.softmax(logits / 100)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[0.14810557 0.14959406 0.1510975  0.15261606 0.39858682]


array([32, 30, 35, 33, 70])

In [53]:
probs = scipy.special.softmax(logits / 0.1)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[0. 0. 0. 0. 1.]


array([  0,   0,   0,   0, 200])

*   logit 들을 1보다 큰 수로 나눈 후 softmax 함수로 확률을 계산하면, logit 값이 작은 토큰들의 확률이 커지는 효과.
    *   토큰을 선택하는 다양성이 커짐.
    *   생성되는 텍스트가 다양해짐.
*   logit 값을 1보다 작은 수로 나눈 후 softmax 함수로 확률을 계산하면, logit 값이 큰 토큰들의 확률을 더 크게 만드는 효과
    *   토큰을 선택하는 다양성이 작아짐.
    *   거의 비슷한 텍스트들이 생성됨.
*   LLM에서 온도(temperature)는 logit 값들을 나눠주는 수.

# top-k 샘플링

모델이 계산한 logit(점수) 값들을 정렬해서 최상위 k개의 토큰들 중에서 선택하는 방법.

k값이 클 수록 더 다양한 텍스트들이 생성될 수 있음.

In [55]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=1.5,
              top_k=100)
print(result)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '네, 맞습니다! 다이어리가 올해 들어 업데이트될 때 내년의 공휴일 정보도 함께 반영될 가능성이 높아요. 하지만 정확한 정보를 확인하시려면직접 **[쇼핑몰 이름] 고객 센터**에 문의 주일 추천드립니다. 그들녘 신속하고 정확하게 도와드릴 수 있을 거예요! 화이팅! 📅💡 서비스 바랍니다 🛒'}]


In [57]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=1.5,
              top_k=10)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '네, 맞습니다! 다이어리에는 주로 매년의 주요 일정과 공휴일들이 정리되어 있지만, **정확히 내년의 공휴일이 포함되었는지 확인하고 싶으시다면, 직접 제품 정보나 고객 서비스에 문의하시는 게 가장 확실합니다.** 저희가 항상 가장 최신 정보를 제공하려고 노력하지만, 각 해가 다르게 공휴일이 정해지기 때문에 확인이 필요한 경우에는 직접 해보시는 것을 권장드립니다! 감사합니다. 😊'}]


# top-p 샘플링

모델이 출력한 logit 값으로 계산한 확률의 크기 순으로 정렬했을 때 누적 확률 p까지의 토큰을 선택하는 방법.

top_p(누적 확률) 값이 클 수록 다양한 텍스트, 작을 수록 일관된 텍스트들이 생성됨.

누적 확률 p가 동일하더라도 선택될 수 있는 토큰의 개수가 달라질 수 있음.

top-p 샘플링은 다양한 확률 분포에 유연하게 대처할 수 있음.

top-k 샘플링은 선택하는 토큰들의 개수를 고정하기 때문에, 높은 확률을 가진 토큰이 선택에서 제외될 가능성이 있음.

In [59]:
result = pipe(messages,
              max_new_tokens=200, return_full_text=False,
              temperature=1.5,
              top_p=0.9)
print(result)

Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '네, 감사합니다! 해당 다이어리에 내년의 공휴일이 상세하게 표기되어 있는지 궁금하시다니 정말 중요한 내용인 것 같네요. 현재로선 정확한 확인은 저희가 제공하기 어려우므로, Product Team으로부터 최신 정보와 답변을 받는 게 좋을 것 같아요. 팀분들이 내년의 모든 공휴일 일정을 확인하고 제품에 반영하는 데 필요한 시간이 조금 필요하니까요. 곧 자세히 알려드릴게요! 궁금하시면 편하게 문의해 주세요. 😊'}]


In [60]:
result = pipe(messages,
              max_new_tokens=200, return_full_text=False,
              temperature=1.5,
              top_p=0.3)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 미리 표기되어 있는지에 대해 궁금하시군요. 제품 담당자가 바로 확인해 주시고 정확한 답변을 드릴 수 있도록 시간을 좀 주세요. 곧 답변 드리겠습니다! 궁금한 점이 더 있으시면 언제든지 말씀해 주세요. 감사합니다. 😊'}]


# Runpod 환경

아래는 Runpod 환경에서 이뤄진 수업의 코드 내용들

In [1]:
# 설치된 파이썬 버전 확인
!python --version

Python 3.12.13


In [2]:
# 설치된 파이썬 패키지 목록 확인
!pip list

Package                                  Version
---------------------------------------- -------------------
absl-py                                  1.4.0
accelerate                               1.13.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                         

In [3]:
# 파이썬 패키지 관리자 pip 업데이트
!python -m pip install -U pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [4]:
# transformers 패키지 설치
!pip install -U transformers==5.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 19.1 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [5]:
import transformers

In [6]:
transformers.__version__

'5.1.0'

In [7]:
# transformers 패키지의 Pipeline 객체 생성
pipe = transformers.pipeline(
    task='text-generation',
    model='LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct',
    device=0,  # GPU 사용
    trust_remote_code=True
)

config.json: 0.00B [00:00, ?B/s]

configuration_exaone.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_exaone.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

In [8]:
# 메시지 탬플릿 생성 - dict들의 list
messages = [
    {
        'role': 'system',
        'content': '너는 아주 유능한 시사평론가야. 현재의 정치/경제 평론을 해줘.'
    },
    {
        'role': 'user',
        'content': '2026 이란-미국/이스라엘 전쟁에 대해서 알려줘.'
    }
]

In [9]:
# LLM 모델에게 메시지를 입력하고 생성된 텍스트를 리턴.
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=True,
              do_sample=True,
              temperature=1.5,
              top_p=0.8)

Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [9]:
result

In [10]:
print(result[0]['generated_text'][-1]['content'])

이란과 미국, 그리고 이스라엘 간의 갈등은 현재 국제정치의 핵심적인 불확실성 중 하나로 부상하고 있으며, 특히 2026년의 전쟁 시나리오는 여러 복잡한 요소들로 인해 예측하기 어렵고 잠재적인 결과에 대해 매우 우려스러운 가능성들을 내포하고 있습니다. 아래는 이러한 긴장 상황을 고려한 평론 내용입니다:

### 배경 및 요인
1. **핵문제**: 이란의 핵무기 개발 계획은 가장 큰 논쟁점입니다. 미국과 서방 국가들은 이란의 핵무기 능력이 지역 안정성과 세계 평화에 심각한 위협이 된다고 우려하고 있습니다. 반면 이란은 자체 핵프로그램을 국제적 안보와 자원 접근성을 확보하기 위한 수단으로 보는 입장입니다.

2. **지역 영향력**: 이스라엘은 시리아와 레바논에서 지속적인 군사 작전을 진행 중이며, 이란은 중동에서의 영향력 확대를 위해 시리아 내 반군 지지와 같은
